# Pro Kabaddi League Data Analysis

Datasets:
1. ALL_Defensive_Skills_Merged.csv
2. ALL_Raider_Skills_Merged.csv
3. AllSeasons_AllTeams_DefenderSuccessRate.csv
4. AllSeasons_AllTeams_RaiderSuccessRate.csv
5. PKL_AggregatedTeamStats.csv
6. all_seasons_player_stats_rounded.csv
7. pkl_matches_s1_s10.csv
8. pkl_player_match_points.csv


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
print("Libraries imported!")

## 2. Find the Kaggle data folder and load all eight CSVs

In [ ]:
INPUT_DIR = Path("/kaggle/input")
found = list(INPUT_DIR.rglob("pkl_matches_s1_s10.csv"))
if not found:
    raise FileNotFoundError("Attach the Pro Kabaddi dataset with Kaggle's Add Input button.")
DATA_DIR = found[0].parent
print("Dataset folder:", DATA_DIR)

defense_skills = pd.read_csv(DATA_DIR / "ALL_Defensive_Skills_Merged.csv")
raider_skills = pd.read_csv(DATA_DIR / "ALL_Raider_Skills_Merged.csv")
defender_success_rate = pd.read_csv(DATA_DIR / "AllSeasons_AllTeams_DefenderSuccessRate.csv")
raider_success_rate = pd.read_csv(DATA_DIR / "AllSeasons_AllTeams_RaiderSuccessRate.csv")
team_stats = pd.read_csv(DATA_DIR / "PKL_AggregatedTeamStats.csv")
player_stats = pd.read_csv(DATA_DIR / "all_seasons_player_stats_rounded.csv")
matches = pd.read_csv(DATA_DIR / "pkl_matches_s1_s10.csv")
player_points = pd.read_csv(DATA_DIR / "pkl_player_match_points.csv")

datasets = {
    "Defensive Skills": defense_skills,
    "Raider Skills": raider_skills,
    "Defender Success Rate": defender_success_rate,
    "Raider Success Rate": raider_success_rate,
    "Aggregated Team Stats": team_stats,
    "Player Stats": player_stats,
    "Matches": matches,
    "Player Match Points": player_points
}
print("All eight datasets loaded.")

## 3. Dataset overview

In [ ]:
for name, df in datasets.items():
    print("\\n" + "=" * 70)
    print(name, "| Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Data types:")
    print(df.dtypes)
    print("Missing values:")
    display(df.isnull().sum().to_frame("Missing values"))
    print("First five rows:")
    display(df.head())

## 4. Reusable helper functions

In [ ]:
def find_col(df, candidates):
    # Find a column by matching its name without worrying about capital letters.
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

def clean_percent(series):
    # Turn text such as '51%' into numeric 51.
    return pd.to_numeric(
        series.astype(str).str.replace("%", "", regex=False).str.strip(),
        errors="coerce"
    )

def bar_chart(series, title, xlabel, ylabel, horizontal=False):
    plt.figure(figsize=(11, 5))
    if horizontal:
        series.sort_values().plot(kind="barh")
    else:
        series.plot(kind="bar")
        plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

print("Helper functions ready.")

# Dataset 1 – Defensive Skills
**File:** `ALL_Defensive_Skills_Merged.csv`

The known layout has descriptive columns first and team columns from index 4. The values are recorded skill counts/values, not player success rates.

**Five questions**
1. Team with highest total recorded defensive-skill value.
2. Team with lowest total recorded defensive-skill value.
3. Defensive skill with highest total recorded value.
4. Season with highest total recorded value.
5. Top recorded defensive skill for each team.

In [ ]:
defense_team_cols = defense_skills.columns[4:].tolist()
for col in defense_team_cols:
    defense_skills[col] = pd.to_numeric(defense_skills[col], errors="coerce")
print("Team columns:", defense_team_cols)
display(defense_skills.head())

### Q1. Which team has the highest total recorded defensive-skill value?

In [ ]:
team_defense_total = defense_skills[defense_team_cols].sum().sort_values(ascending=False)
display(team_defense_total.to_frame("Total recorded value"))
print("Highest team:", team_defense_total.idxmax())
print("Highest total:", team_defense_total.max())
bar_chart(team_defense_total, "Total Defensive-Skill Value by Team", "Team", "Recorded value")

### Q2. Which team has the lowest total recorded defensive-skill value?

In [ ]:
print("Lowest team:", team_defense_total.idxmin())
print("Lowest total:", team_defense_total.min())
display(team_defense_total.sort_values().to_frame("Total recorded value"))

### Q3. Which defensive skill has the highest total recorded value?

In [ ]:
skill_col = find_col(defense_skills, ["Skill Name"])
if skill_col:
    defense_skill_total = defense_skills.groupby(skill_col)[defense_team_cols].sum().sum(axis=1).sort_values(ascending=False)
    display(defense_skill_total.to_frame("Total recorded value"))
    print("Top skill:", defense_skill_total.idxmax())
    bar_chart(defense_skill_total.head(10), "Top 10 Defensive Skills", "Skill", "Recorded value", True)
else:
    print("Skill Name column not found. Check the column list.")

### Q4. Which season has the highest total recorded defensive-skill value?

In [ ]:
season_col = find_col(defense_skills, ["Season", "season"])
if season_col:
    defense_season_total = defense_skills.groupby(season_col)[defense_team_cols].sum().sum(axis=1).sort_index()
    display(defense_season_total.to_frame("Total recorded value"))
    print("Highest season:", defense_season_total.idxmax())
    bar_chart(defense_season_total, "Defensive-Skill Value by Season", "Season", "Recorded value")
else:
    print("Season column not found.")

### Q5. Which defensive skill has the highest total for each team?

In [ ]:
if skill_col:
    skill_team_table = defense_skills.groupby(skill_col)[defense_team_cols].sum()
    display(skill_team_table.idxmax().to_frame("Top skill for team"))
else:
    print("Skill Name column not found.")

# Dataset 2 – Raider Skills
**File:** `ALL_Raider_Skills_Merged.csv`

The known layout has descriptive columns first and team columns from index 3. We analyse recorded skill values and do not assume that `Skill Type` contains a category named `Successful Raids`.

**Five questions**
1. Team with highest total recorded raiding-skill value.
2. Team with lowest total recorded raiding-skill value.
3. Raiding skill with highest total recorded value.
4. Season with highest total recorded value.
5. Top recorded raiding skill for each team.

In [ ]:
raider_team_cols = raider_skills.columns[3:].tolist()
for col in raider_team_cols:
    raider_skills[col] = pd.to_numeric(raider_skills[col], errors="coerce")
print("Team columns:", raider_team_cols)
print("Skill Type values:", raider_skills["Skill Type"].dropna().unique())
print("Skill Name values:", raider_skills["Skill Name"].dropna().unique())
display(raider_skills.head())

### Q1. Which team has the highest total recorded raiding-skill value?

In [ ]:
team_raider_total = raider_skills[raider_team_cols].sum().sort_values(ascending=False)
display(team_raider_total.to_frame("Total recorded value"))
print("Highest team:", team_raider_total.idxmax())
print("Highest total:", team_raider_total.max())
bar_chart(team_raider_total, "Total Raiding-Skill Value by Team", "Team", "Recorded value")

### Q2. Which team has the lowest total recorded raiding-skill value?

In [ ]:
print("Lowest team:", team_raider_total.idxmin())
print("Lowest total:", team_raider_total.min())
display(team_raider_total.sort_values().to_frame("Total recorded value"))

### Q3. Which raiding skill has the highest total recorded value?

In [ ]:
raider_skill_col = find_col(raider_skills, ["Skill Name"])
if raider_skill_col:
    raider_skill_total = raider_skills.groupby(raider_skill_col)[raider_team_cols].sum().sum(axis=1).sort_values(ascending=False)
    display(raider_skill_total.to_frame("Total recorded value"))
    print("Top skill:", raider_skill_total.idxmax())
    bar_chart(raider_skill_total.head(10), "Top 10 Raider Skills", "Skill", "Recorded value", True)
else:
    print("Skill Name column not found.")

### Q4. Which season has the highest total recorded raiding-skill value?

In [ ]:
raider_season_col = find_col(raider_skills, ["Season", "season"])
if raider_season_col:
    raider_season_total = raider_skills.groupby(raider_season_col)[raider_team_cols].sum().sum(axis=1).sort_index()
    display(raider_season_total.to_frame("Total recorded value"))
    print("Highest season:", raider_season_total.idxmax())
    bar_chart(raider_season_total, "Raiding-Skill Value by Season", "Season", "Recorded value")
else:
    print("Season column not found.")

### Q5. Which raiding skill has the highest total for each team?

In [ ]:
if raider_skill_col:
    raider_skill_team_table = raider_skills.groupby(raider_skill_col)[raider_team_cols].sum()
    display(raider_skill_team_table.idxmax().to_frame("Top skill for team"))
else:
    print("Skill Name column not found.")

# Dataset – Defender Success Rate
**File:** `AllSeasons_AllTeams_DefenderSuccessRate.csv`

First inspect the detected columns. If attempts and successful attempts are available, the notebook calculates a **weighted success rate**: total successful attempts divided by total attempts, multiplied by 100. This is usually more representative than taking a simple average of player percentages.

**Five questions**
1. Team with highest overall success rate.
2. Team with lowest overall success rate.
3. Season with highest overall success rate.
4. Team that improved the most from its first to last available season.
5. Player with highest success rate, if player-level columns exist.

In [ ]:
print(defender_success_rate.columns.tolist())
display(defender_success_rate.head())

team_col = find_col(defender_success_rate, ["Team", "team"])
season_col = find_col(defender_success_rate, ["season", "Season"])
player_col = find_col(defender_success_rate, ["Defender Name", "Player Name", "player_name", "Name"])
attempt_col = find_col(defender_success_rate, ["Total Tackles", "Total Raids", "Total Raid", "Total Attempts"])
success_col = find_col(defender_success_rate, ["Successful Tackles", "Successful Raids", "Successful Raid", "Successful Attempts"])
rate_col = find_col(defender_success_rate, ["Defender Success rate", "Defender Success Rate", "Success Rate", "Success rate"])

print("Team:", team_col, "| Season:", season_col, "| Player:", player_col)
print("Attempts:", attempt_col, "| Successful:", success_col, "| Rate:", rate_col)

if rate_col:
    defender_success_rate[rate_col] = clean_percent(defender_success_rate[rate_col])
for col in [attempt_col, success_col]:
    if col:
        defender_success_rate[col] = pd.to_numeric(defender_success_rate[col], errors="coerce")

### Q1. Which team has the highest overall success rate?

In [ ]:
if team_col and attempt_col and success_col:
    team_summary = defender_success_rate.groupby(team_col)[[attempt_col, success_col]].sum()
    team_summary["Success Rate"] = team_summary[success_col] / team_summary[attempt_col].replace(0, np.nan) * 100
elif team_col and rate_col:
    team_summary = defender_success_rate.groupby(team_col)[rate_col].mean().to_frame("Success Rate")
else:
    team_summary = pd.DataFrame()

if not team_summary.empty:
    team_summary = team_summary.sort_values("Success Rate", ascending=False)
    display(team_summary)
    print("Highest team:", team_summary["Success Rate"].idxmax())
    print("Highest rate:", round(team_summary["Success Rate"].max(), 2), "%")
    bar_chart(team_summary["Success Rate"], "Defender Success Rate: Success Rate by Team", "Team", "Success rate (%)")
else:
    print("Could not identify team plus rate or attempt/success columns. Inspect the printed column names.")

### Q2. Which team has the lowest overall success rate?

In [ ]:
if not team_summary.empty:
    print("Lowest team:", team_summary["Success Rate"].idxmin())
    print("Lowest rate:", round(team_summary["Success Rate"].min(), 2), "%")
    display(team_summary.sort_values("Success Rate").head(10))
else:
    print("Team summary is unavailable; check the columns.")

### Q3. Which season has the highest overall success rate?

In [ ]:
if season_col and attempt_col and success_col:
    season_summary = defender_success_rate.groupby(season_col)[[attempt_col, success_col]].sum()
    season_summary["Success Rate"] = season_summary[success_col] / season_summary[attempt_col].replace(0, np.nan) * 100
    season_summary = season_summary.sort_values("Success Rate", ascending=False)
    display(season_summary)
    print("Highest season:", season_summary["Success Rate"].idxmax())
elif season_col and rate_col:
    season_summary = defender_success_rate.groupby(season_col)[rate_col].mean().sort_values(ascending=False)
    display(season_summary.to_frame("Average success rate"))
    print("Highest season:", season_summary.idxmax())
else:
    print("Need season and rate, or season plus attempts and successful attempts.")

### Q4. Which team improved the most from its first to last available season?

In [ ]:
if team_col and season_col and attempt_col and success_col:
    trend = defender_success_rate.groupby([team_col, season_col])[[attempt_col, success_col]].sum().reset_index()
    trend["Success Rate"] = trend[success_col] / trend[attempt_col].replace(0, np.nan) * 100
    trend = trend.sort_values([team_col, season_col])
    first_rate = trend.groupby(team_col).first()["Success Rate"]
    last_rate = trend.groupby(team_col).last()["Success Rate"]
    improvement = (last_rate - first_rate).sort_values(ascending=False)
    display(improvement.to_frame("Improvement in percentage points"))
    print("Most improved team:", improvement.idxmax())
elif team_col and season_col and rate_col:
    trend = defender_success_rate.groupby([team_col, season_col])[rate_col].mean().reset_index().sort_values([team_col, season_col])
    first_rate = trend.groupby(team_col).first()[rate_col]
    last_rate = trend.groupby(team_col).last()[rate_col]
    improvement = (last_rate - first_rate).sort_values(ascending=False)
    display(improvement.to_frame("Improvement in percentage points"))
    print("Most improved team:", improvement.idxmax())
else:
    print("Need team, season and rate or attempt/success columns.")

### Q5. Which player has the highest success rate?

In [ ]:
if player_col and attempt_col and success_col:
    player_summary = defender_success_rate.groupby(player_col)[[attempt_col, success_col]].sum()
    player_summary["Success Rate"] = player_summary[success_col] / player_summary[attempt_col].replace(0, np.nan) * 100
    player_summary = player_summary.sort_values("Success Rate", ascending=False)
    display(player_summary.head(10))
    if not player_summary.empty:
        print("Top player:", player_summary["Success Rate"].idxmax())
elif player_col and rate_col:
    player_summary = defender_success_rate.groupby(player_col)[rate_col].mean().sort_values(ascending=False)
    display(player_summary.head(10).to_frame("Average success rate"))
    if not player_summary.empty:
        print("Top player:", player_summary.idxmax())
else:
    print("Player-level success rate could not be calculated from the detected columns.")

# Dataset – Raider Success Rate
**File:** `AllSeasons_AllTeams_RaiderSuccessRate.csv`

First inspect the detected columns. If attempts and successful attempts are available, the notebook calculates a **weighted success rate**: total successful attempts divided by total attempts, multiplied by 100. This is usually more representative than taking a simple average of player percentages.

**Five questions**
1. Team with highest overall success rate.
2. Team with lowest overall success rate.
3. Season with highest overall success rate.
4. Team that improved the most from its first to last available season.
5. Player with highest success rate, if player-level columns exist.

In [ ]:
print(raider_success_rate.columns.tolist())
display(raider_success_rate.head())

team_col = find_col(raider_success_rate, ["Team", "team"])
season_col = find_col(raider_success_rate, ["season", "Season"])
player_col = find_col(raider_success_rate, ["Raider Name", "Player Name", "player_name", "Name"])
attempt_col = find_col(raider_success_rate, ["Total Tackles", "Total Raids", "Total Raid", "Total Attempts"])
success_col = find_col(raider_success_rate, ["Successful Tackles", "Successful Raids", "Successful Raid", "Successful Attempts"])
rate_col = find_col(raider_success_rate, ["Raider Success rate", "Raider Success Rate", "Success Rate", "Success rate"])

print("Team:", team_col, "| Season:", season_col, "| Player:", player_col)
print("Attempts:", attempt_col, "| Successful:", success_col, "| Rate:", rate_col)

if rate_col:
    raider_success_rate[rate_col] = clean_percent(raider_success_rate[rate_col])
for col in [attempt_col, success_col]:
    if col:
        raider_success_rate[col] = pd.to_numeric(raider_success_rate[col], errors="coerce")

### Q1. Which team has the highest overall success rate?

In [ ]:
if team_col and attempt_col and success_col:
    team_summary = raider_success_rate.groupby(team_col)[[attempt_col, success_col]].sum()
    team_summary["Success Rate"] = team_summary[success_col] / team_summary[attempt_col].replace(0, np.nan) * 100
elif team_col and rate_col:
    team_summary = raider_success_rate.groupby(team_col)[rate_col].mean().to_frame("Success Rate")
else:
    team_summary = pd.DataFrame()

if not team_summary.empty:
    team_summary = team_summary.sort_values("Success Rate", ascending=False)
    display(team_summary)
    print("Highest team:", team_summary["Success Rate"].idxmax())
    print("Highest rate:", round(team_summary["Success Rate"].max(), 2), "%")
    bar_chart(team_summary["Success Rate"], "Raider Success Rate: Success Rate by Team", "Team", "Success rate (%)")
else:
    print("Could not identify team plus rate or attempt/success columns. Inspect the printed column names.")

### Q2. Which team has the lowest overall success rate?

In [ ]:
if not team_summary.empty:
    print("Lowest team:", team_summary["Success Rate"].idxmin())
    print("Lowest rate:", round(team_summary["Success Rate"].min(), 2), "%")
    display(team_summary.sort_values("Success Rate").head(10))
else:
    print("Team summary is unavailable; check the columns.")

### Q3. Which season has the highest overall success rate?

In [ ]:
if season_col and attempt_col and success_col:
    season_summary = raider_success_rate.groupby(season_col)[[attempt_col, success_col]].sum()
    season_summary["Success Rate"] = season_summary[success_col] / season_summary[attempt_col].replace(0, np.nan) * 100
    season_summary = season_summary.sort_values("Success Rate", ascending=False)
    display(season_summary)
    print("Highest season:", season_summary["Success Rate"].idxmax())
elif season_col and rate_col:
    season_summary = raider_success_rate.groupby(season_col)[rate_col].mean().sort_values(ascending=False)
    display(season_summary.to_frame("Average success rate"))
    print("Highest season:", season_summary.idxmax())
else:
    print("Need season and rate, or season plus attempts and successful attempts.")

### Q4. Which team improved the most from its first to last available season?

In [ ]:
if team_col and season_col and attempt_col and success_col:
    trend = raider_success_rate.groupby([team_col, season_col])[[attempt_col, success_col]].sum().reset_index()
    trend["Success Rate"] = trend[success_col] / trend[attempt_col].replace(0, np.nan) * 100
    trend = trend.sort_values([team_col, season_col])
    first_rate = trend.groupby(team_col).first()["Success Rate"]
    last_rate = trend.groupby(team_col).last()["Success Rate"]
    improvement = (last_rate - first_rate).sort_values(ascending=False)
    display(improvement.to_frame("Improvement in percentage points"))
    print("Most improved team:", improvement.idxmax())
elif team_col and season_col and rate_col:
    trend = raider_success_rate.groupby([team_col, season_col])[rate_col].mean().reset_index().sort_values([team_col, season_col])
    first_rate = trend.groupby(team_col).first()[rate_col]
    last_rate = trend.groupby(team_col).last()[rate_col]
    improvement = (last_rate - first_rate).sort_values(ascending=False)
    display(improvement.to_frame("Improvement in percentage points"))
    print("Most improved team:", improvement.idxmax())
else:
    print("Need team, season and rate or attempt/success columns.")

### Q5. Which player has the highest success rate?

In [ ]:
if player_col and attempt_col and success_col:
    player_summary = raider_success_rate.groupby(player_col)[[attempt_col, success_col]].sum()
    player_summary["Success Rate"] = player_summary[success_col] / player_summary[attempt_col].replace(0, np.nan) * 100
    player_summary = player_summary.sort_values("Success Rate", ascending=False)
    display(player_summary.head(10))
    if not player_summary.empty:
        print("Top player:", player_summary["Success Rate"].idxmax())
elif player_col and rate_col:
    player_summary = raider_success_rate.groupby(player_col)[rate_col].mean().sort_values(ascending=False)
    display(player_summary.head(10).to_frame("Average success rate"))
    if not player_summary.empty:
        print("Top player:", player_summary.idxmax())
else:
    print("Player-level success rate could not be calculated from the detected columns.")

# Dataset – Aggregated Team Stats
**File:** `PKL_AggregatedTeamStats.csv`

This dataset has a different structure from the skill and success-rate files. The notebook prints its column names and automatically chooses a likely team/player/match label and a numeric statistic. **Check these selected columns** before interpreting the results.

**Five questions**
1. Which team has the highest total of the selected statistic?
2. Which team has the lowest total?
3. Which team has the highest average?
4. What are the top 10 rows by the selected statistic?
5. Which numeric columns have the strongest relationship (correlation)?

In [ ]:
print(team_stats.columns.tolist())
display(team_stats.head())

numeric_cols = team_stats.select_dtypes(include="number").columns.tolist()
text_cols = team_stats.select_dtypes(include=["object", "category"]).columns.tolist()

group_col = find_col(team_stats, ["Team", "Team Name", "Player", "Player Name",
                             "Defender Name", "Raider Name", "Match", "match_id"])
if group_col is None and text_cols:
    group_col = text_cols[0]
metric_col = numeric_cols[0] if numeric_cols else None

print("Numeric columns:", numeric_cols)
print("Text columns:", text_cols)
print("Selected group column:", group_col)
print("Selected numeric statistic:", metric_col)

### Q1. Which team has the highest total?

In [ ]:
if group_col and metric_col:
    totals = team_stats.groupby(group_col)[metric_col].sum().sort_values(ascending=False)
    display(totals.head(10).to_frame("Total " + metric_col))
    print("Highest:", totals.idxmax(), "=", totals.max())
    bar_chart(totals.head(10), "Top 10 by total " + metric_col, group_col, metric_col, True)
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q2. Which team has the lowest total?

In [ ]:
if group_col and metric_col:
    low_totals = totals.sort_values()
    display(low_totals.head(10).to_frame("Total " + metric_col))
    print("Lowest:", low_totals.idxmin(), "=", low_totals.min())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q3. Which team has the highest average?

In [ ]:
if group_col and metric_col:
    averages = team_stats.groupby(group_col)[metric_col].mean().sort_values(ascending=False)
    display(averages.head(10).to_frame("Average " + metric_col))
    print("Highest average:", averages.idxmax(), "=", averages.max())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q4. What are the top 10 rows by the selected numeric statistic?

In [ ]:
if metric_col:
    top_rows = team_stats.sort_values(metric_col, ascending=False).head(10)
    display(top_rows)
else:
    print("No numeric column was detected.")

### Q5. Which numeric columns have the strongest relationship?

In [ ]:
if len(numeric_cols) >= 2:
    corr = team_stats[numeric_cols].corr(numeric_only=True)
    display(corr)
    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation between numeric columns")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are required for correlation.")

# Dataset – Player Stats
**File:** `all_seasons_player_stats_rounded.csv`

This dataset has a different structure from the skill and success-rate files. The notebook prints its column names and automatically chooses a likely team/player/match label and a numeric statistic. **Check these selected columns** before interpreting the results.

**Five questions**
1. Which player has the highest total of the selected statistic?
2. Which player has the lowest total?
3. Which player has the highest average?
4. What are the top 10 rows by the selected statistic?
5. Which numeric columns have the strongest relationship (correlation)?

In [ ]:
print(player_stats.columns.tolist())
display(player_stats.head())

numeric_cols = player_stats.select_dtypes(include="number").columns.tolist()
text_cols = player_stats.select_dtypes(include=["object", "category"]).columns.tolist()

group_col = find_col(player_stats, ["Team", "Team Name", "Player", "Player Name",
                             "Defender Name", "Raider Name", "Match", "match_id"])
if group_col is None and text_cols:
    group_col = text_cols[0]
metric_col = numeric_cols[0] if numeric_cols else None

print("Numeric columns:", numeric_cols)
print("Text columns:", text_cols)
print("Selected group column:", group_col)
print("Selected numeric statistic:", metric_col)

### Q1. Which player has the highest total?

In [ ]:
if group_col and metric_col:
    totals = player_stats.groupby(group_col)[metric_col].sum().sort_values(ascending=False)
    display(totals.head(10).to_frame("Total " + metric_col))
    print("Highest:", totals.idxmax(), "=", totals.max())
    bar_chart(totals.head(10), "Top 10 by total " + metric_col, group_col, metric_col, True)
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q2. Which player has the lowest total?

In [ ]:
if group_col and metric_col:
    low_totals = totals.sort_values()
    display(low_totals.head(10).to_frame("Total " + metric_col))
    print("Lowest:", low_totals.idxmin(), "=", low_totals.min())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q3. Which player has the highest average?

In [ ]:
if group_col and metric_col:
    averages = player_stats.groupby(group_col)[metric_col].mean().sort_values(ascending=False)
    display(averages.head(10).to_frame("Average " + metric_col))
    print("Highest average:", averages.idxmax(), "=", averages.max())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q4. What are the top 10 rows by the selected numeric statistic?

In [ ]:
if metric_col:
    top_rows = player_stats.sort_values(metric_col, ascending=False).head(10)
    display(top_rows)
else:
    print("No numeric column was detected.")

### Q5. Which numeric columns have the strongest relationship?

In [ ]:
if len(numeric_cols) >= 2:
    corr = player_stats[numeric_cols].corr(numeric_only=True)
    display(corr)
    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation between numeric columns")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are required for correlation.")

# Dataset – Matches
**File:** `pkl_matches_s1_s10.csv`

This dataset has a different structure from the skill and success-rate files. The notebook prints its column names and automatically chooses a likely team/player/match label and a numeric statistic. **Check these selected columns** before interpreting the results.

**Five questions**
1. Which match/team has the highest total of the selected statistic?
2. Which match/team has the lowest total?
3. Which match/team has the highest average?
4. What are the top 10 rows by the selected statistic?
5. Which numeric columns have the strongest relationship (correlation)?

In [ ]:
print(matches.columns.tolist())
display(matches.head())

numeric_cols = matches.select_dtypes(include="number").columns.tolist()
text_cols = matches.select_dtypes(include=["object", "category"]).columns.tolist()

group_col = find_col(matches, ["Team", "Team Name", "Player", "Player Name",
                             "Defender Name", "Raider Name", "Match", "match_id"])
if group_col is None and text_cols:
    group_col = text_cols[0]
metric_col = numeric_cols[0] if numeric_cols else None

print("Numeric columns:", numeric_cols)
print("Text columns:", text_cols)
print("Selected group column:", group_col)
print("Selected numeric statistic:", metric_col)

### Q1. Which match/team has the highest total?

In [ ]:
if group_col and metric_col:
    totals = matches.groupby(group_col)[metric_col].sum().sort_values(ascending=False)
    display(totals.head(10).to_frame("Total " + metric_col))
    print("Highest:", totals.idxmax(), "=", totals.max())
    bar_chart(totals.head(10), "Top 10 by total " + metric_col, group_col, metric_col, True)
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q2. Which match/team has the lowest total?

In [ ]:
if group_col and metric_col:
    low_totals = totals.sort_values()
    display(low_totals.head(10).to_frame("Total " + metric_col))
    print("Lowest:", low_totals.idxmin(), "=", low_totals.min())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q3. Which match/team has the highest average?

In [ ]:
if group_col and metric_col:
    averages = matches.groupby(group_col)[metric_col].mean().sort_values(ascending=False)
    display(averages.head(10).to_frame("Average " + metric_col))
    print("Highest average:", averages.idxmax(), "=", averages.max())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q4. What are the top 10 rows by the selected numeric statistic?

In [ ]:
if metric_col:
    top_rows = matches.sort_values(metric_col, ascending=False).head(10)
    display(top_rows)
else:
    print("No numeric column was detected.")

### Q5. Which numeric columns have the strongest relationship?

In [ ]:
if len(numeric_cols) >= 2:
    corr = matches[numeric_cols].corr(numeric_only=True)
    display(corr)
    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation between numeric columns")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are required for correlation.")

# Dataset – Player Match Points
**File:** `pkl_player_match_points.csv`

This dataset has a different structure from the skill and success-rate files. The notebook prints its column names and automatically chooses a likely team/player/match label and a numeric statistic. **Check these selected columns** before interpreting the results.

**Five questions**
1. Which player/match has the highest total of the selected statistic?
2. Which player/match has the lowest total?
3. Which player/match has the highest average?
4. What are the top 10 rows by the selected statistic?
5. Which numeric columns have the strongest relationship (correlation)?

In [ ]:
print(player_points.columns.tolist())
display(player_points.head())

numeric_cols = player_points.select_dtypes(include="number").columns.tolist()
text_cols = player_points.select_dtypes(include=["object", "category"]).columns.tolist()

group_col = find_col(player_points, ["Team", "Team Name", "Player", "Player Name",
                             "Defender Name", "Raider Name", "Match", "match_id"])
if group_col is None and text_cols:
    group_col = text_cols[0]
metric_col = numeric_cols[0] if numeric_cols else None

print("Numeric columns:", numeric_cols)
print("Text columns:", text_cols)
print("Selected group column:", group_col)
print("Selected numeric statistic:", metric_col)

### Q1. Which player/match has the highest total?

In [ ]:
if group_col and metric_col:
    totals = player_points.groupby(group_col)[metric_col].sum().sort_values(ascending=False)
    display(totals.head(10).to_frame("Total " + metric_col))
    print("Highest:", totals.idxmax(), "=", totals.max())
    bar_chart(totals.head(10), "Top 10 by total " + metric_col, group_col, metric_col, True)
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q2. Which player/match has the lowest total?

In [ ]:
if group_col and metric_col:
    low_totals = totals.sort_values()
    display(low_totals.head(10).to_frame("Total " + metric_col))
    print("Lowest:", low_totals.idxmin(), "=", low_totals.min())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q3. Which player/match has the highest average?

In [ ]:
if group_col and metric_col:
    averages = player_points.groupby(group_col)[metric_col].mean().sort_values(ascending=False)
    display(averages.head(10).to_frame("Average " + metric_col))
    print("Highest average:", averages.idxmax(), "=", averages.max())
else:
    print("A suitable grouping column and numeric statistic are required.")

### Q4. What are the top 10 rows by the selected numeric statistic?

In [ ]:
if metric_col:
    top_rows = player_points.sort_values(metric_col, ascending=False).head(10)
    display(top_rows)
else:
    print("No numeric column was detected.")

### Q5. Which numeric columns have the strongest relationship?

In [ ]:
if len(numeric_cols) >= 2:
    corr = player_points[numeric_cols].corr(numeric_only=True)
    display(corr)
    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation between numeric columns")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are required for correlation.")

# 5. Final checklist for your portfolio

- Run all cells from top to bottom.
- Check automatically selected column names before trusting the output.
- Distinguish counts, averages, percentages and weighted success rates.
- For success rates, prefer `sum(successful attempts) / sum(total attempts) * 100` when both columns exist.
- Avoid comparing raw totals without considering matches played or opportunities.
- Add the actual result, chart and a short interpretation to your GitHub README.
- Mention limitations such as missing data or different seasons.

### How to write each finding
**Question:** Write the analysis question.

**Result:** Copy the team/player/season and value from the notebook output.

**Interpretation:** Explain what the result means in one or two sentences.

**Limitation:** State whether the result is a count, simple average or weighted rate.

**Learning tip:** Try to explain each line of code in your own words. This is excellent practice for data analyst interviews.